# 第一课 · 一张订单在原始数据里是什么样？

运行后，你应该能用自己的话解释「交易明细行」「商品种类」「购买件数」和「购物篮」。本课只读取工作簿，CPU 即可。每段代码都可以点击左侧的 ▶ 单独运行，请从上往下操作。

数据来源为 D. Chen（2015）的 UCI Online Retail，许可为 CC BY 4.0。本课基于课程提供的工作簿，示例金额单位为英镑。[数据来源](https://doi.org/10.24432/C5BW33)

## 1. 找到工作簿

先在 Google Drive 里确认项目文件夹下有 `data/raw/Online Retail.xlsx`。如果文件夹在「与我共享」，可以把它作为快捷方式添加到「我的云端硬盘」。在 Colab 运行下一格，按提示授权挂载 Drive。

代码会先检查项目约定的路径。如果出现输入框，打开 Colab 左栏的「文件」，在 `drive` 下找到 Excel 文件，从它的菜单选择「复制路径」，粘贴到输入框并按回车。输入框接收文件路径。使用本地 Jupyter 时，代码会查找仓库内的数据文件。

In [ ]:
from pathlib import Path

try:
    from google.colab import drive
except ImportError:
    drive = None

if drive is not None:
    drive.mount("/content/drive")
    workbook = Path("/content/drive/MyDrive/CS5481_Project2_OnlineRetail/data/raw/Online Retail.xlsx")
    if not workbook.is_file():
        workbook = Path(input("请粘贴从 Colab「文件」面板复制的 Excel 路径：").strip())
else:
    workbook = Path("data/raw/Online Retail.xlsx")
    if not workbook.is_file():
        workbook = Path("..") / workbook

if not workbook.is_file():
    raise FileNotFoundError("没有找到工作簿。请核对文件名、路径和当前账号的读取权限，再运行本格。")
print("本次读取的文件：", workbook.resolve())

## 2. 确认数据版本，再读入 Excel

SHA-256 是根据文件全部内容算出的校验值。同名文件也可能有不同内容。第一段把实际值和研究计划中的值比较，匹配后才继续。第二段读取名为 `Online Retail` 的工作表，把三个编号列按文本保存，以便保留商品编码里的字母。读取 54 万多行可能需要稍候。

In [ ]:
from hashlib import sha256
import pandas as pd

expected_hash = "43465a06f2ccf7c8b5bd2892bc7defb52f97487934fe93b16ae4c3936424676d"
actual_hash = sha256(workbook.read_bytes()).hexdigest()
print("文件 SHA-256：", actual_hash)
if actual_hash != expected_hash:
    raise ValueError("当前文件与研究计划记录的版本不同。请选择未经编辑的原始工作簿，再从第一步运行。")

identifiers = {"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"}
sales = pd.read_excel(workbook, sheet_name="Online Retail", dtype=identifiers)
print(f"共 {len(sales):,} 行交易明细，{len(sales.columns)} 列。")
print("字段：", ", ".join(sales.columns))
assert sales.shape == (541909, 8), "请检查工作表的行数和列数。"

## 3. 读懂一行，再看整张发票

| 列名 | 看这一列能回答什么？ |
|---|---|
| InvoiceNo | 这行属于哪张发票？ |
| StockCode | 是哪种商品？例如 `85123A`。 |
| Description | 商品的文字描述是什么？ |
| Quantity | 这行记录了多少件？ |
| InvoiceDate | 发票记录的日期和时间是什么？ |
| UnitPrice | 每件多少英镑？ |
| CustomerID | 客户编号是什么？原表中有些行缺少它。 |
| Country | 客户所在的国家或地区是哪里？ |

下面只挑出客户 `17850` 的发票 `536365`。先看第一行，再纵向比较：哪些字段相同？哪些字段随商品变化？

In [ ]:
invoice = sales.loc[
    (sales["CustomerID"] == "17850") & (sales["InvoiceNo"] == "536365")
].copy()
invoice

## 4. 数一数这张购物篮

先猜：这张发票里有几种商品？总共买了几件？第一行的金额是多少？

`len` 数明细行，`nunique` 数不同的商品编码，`sum` 把购买数量加起来。每行先用数量乘以单价，再把行金额加起来，就得到这张发票的总金额。

In [ ]:
line_amount = invoice["Quantity"] * invoice["UnitPrice"]
print("明细行数：", len(invoice))
print("不同商品种数：", invoice["StockCode"].nunique())
print("购买总件数：", invoice["Quantity"].sum())
print(f"第一行金额：£{line_amount.iloc[0]:.2f}")
print(f"整张发票金额：£{line_amount.sum():.2f}")

**读数自检**　商品 `85123A` 的 6 件商品按 £2.55 计价，这一行是 £15.30。整张发票共有 7 行、7 种商品、40 件，合计 £139.12。

**为什么这和推荐有关？** 同一位客户在同一张发票里买的商品构成一个购物篮。这个例子中，7 行恰好对应 7 种商品。想一想：如果同一个商品编码在一张发票里出现两行，统计「买了几种」和「买了几件」时，各该怎么算？